# Tests

In [0]:
print("Hello Databricks")

In [0]:
data = [
    (1, "Aya", "Morocco"),
    (2, "Sara", "France"),
    (3, "Adam", "Germany")
]

df = spark.createDataFrame(
    data,
    ["id", "name", "country"]
)

display(df)

In [0]:
df.printSchema()

In [0]:
df.select("name", "country").show()

In [0]:
df.filter(df.country == "Morocco").show()

In [0]:
df.groupBy("country").count().show()

In [0]:
df.filter(df.country == "Morocco").show()

In [0]:
df.filter(df.id > 1).show()

In [0]:
morocco_df = df.filter(df.country == "Morocco")

In [0]:
display(morocco_df)

In [0]:
display(df)

Python data

    ↓
Spark DataFrame

    ↓
Serverless Spark compute
> 
    ↓
Databricks display

In [0]:
from pyspark.sql.functions import col, upper

df2 = df.withColumn(
    "country_upper",
    upper(col("country"))
)

display(df2)

In [0]:
df2 = df2.drop("country_upper")
display(df2)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("e2e_project.default.people")

In [0]:
%sql

SELECT *
FROM e2e_project.default.people;


**SQL filter**

In [0]:
%sql

SELECT *
FROM e2e_project.default.people
WHERE country = 'Morocco';


**aggregation**

In [0]:
%sql

SELECT
    country,
    COUNT(*) AS number_of_people
FROM e2e_project.default.people
GROUP BY country;

# Create our Medallion Architecture schemas

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS e2e_project.bronze;

CREATE SCHEMA IF NOT EXISTS e2e_project.silver;

CREATE SCHEMA IF NOT EXISTS e2e_project.gold;

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS
e2e_project.bronze.source_files;

we'll build our first real Bronze table

In [0]:
crm_customers = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(
        "/Volumes/e2e_project/bronze/source_files/source_crm/cust_info.csv"
    )
)

display(crm_customers)

Then we'll inspect the data before saving it

In [0]:
crm_customers.printSchema()

In [0]:
crm_customers.show(10, truncate=False)

In [0]:
print("Rows:", crm_customers.count())

This step in order to answer these questions :
What are the columns?
What types did Spark infer?
How many records exist?
Are there NULL values?
Do the values look strange?

At Bronze we won't necessarily correct those problems yet. But we observe them because Silver will need to address them.

Create the real Bronze Delta table

cust_info.csv

      │
      ▼
Spark DataFrame

      │
      ▼
Delta Lake

      │
      ▼
e2e_project.bronze.crm_cust_info

In [0]:
(
    crm_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "e2e_project.bronze.crm_cust_info"
    )
)

Bronze's purpose is approximately:

"What did the source system give us?"

Silver answers:

"What should clean, standardized data look like?"

Gold answers:

"What structure does the business need for analytics?"